# VitroVision รอบ 2 — SAM3 วิเคราะห์ 100 ขวด พริกจินดา
รัน: 18 ส.ค. 2569 · ชุดภาพ 20260814_batch (001–100.jpg) · species = พริกจินดา (Capsicum frutescens)
ไฟล์ข้อมูล: `/content/drive/MyDrive/_staging_20260814_batch.zip` + `sam3_growth_pipeline.py`
รันทุก cell เรียงตามลำดับ (Runtime → Run all)

In [ ]:
# 1) ติดตั้ง dependency
!pip install -q transformers torch torchvision opencv-python pillow matplotlib pandas numpy huggingface_hub xlsxwriter openpyxl
print("deps OK")

In [ ]:
# 2) Login Hugging Face — อ่าน token จาก environment (ห้าม hardcode token ในไฟล์/commit!)
#    วิธีใช้: ใส่ token ใน Colab secrets (HF_TOKEN) แล้วรัน cell นี้
import os
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF login OK")


In [ ]:
# 3) Mount Google Drive (กด Allow ถ้าขึ้น popup)
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

In [ ]:
# 4) เตรียมข้อมูลจาก Drive: copy script + unzip ภาพ + species_map
import os, zipfile, shutil
DRIVE = "/content/drive/MyDrive"
os.makedirs("/content/data", exist_ok=True)
shutil.copy(f"{DRIVE}/sam3_growth_pipeline.py", "/content/sam3_growth_pipeline.py")
with zipfile.ZipFile(f"{DRIVE}/_staging_20260814_batch.zip") as z:
    z.extractall("/content/data")
imgs = sorted(f for f in os.listdir("/content/data") if f.lower().endswith(".jpg"))
print("ภาพ:", len(imgs), "| script:", os.path.exists("/content/sam3_growth_pipeline.py"))

In [ ]:
# 5) รัน pipeline (SAM3 5 prompts + ROI ขวด + verdict 3 คลาส + checkpoint + species summary)
import time
t0 = time.time()
!python /content/sam3_growth_pipeline.py --data /content/data --out /content/results
print(f"RUNTIME_MIN={(time.time()-t0)/60:.1f}")

In [ ]:
# 6) บันทึกผลลง Drive + ดาวน์โหลดกลับเครื่อง
import shutil, os
shutil.make_archive("/content/results_round2", "zip", "/content/results")
shutil.copy("/content/results_round2.zip", "/content/drive/MyDrive/results_round2_20260818.zip")
print("บันทึก Drive: results_round2_20260818.zip")
from google.colab import files
files.download("/content/results_round2.zip")